<a href="https://colab.research.google.com/github/Ammara-Qaisar123/Data-Science-Analytics-P2/blob/main/P_2_Tak_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Data Science & Analytics
##Phase 2
##Submitted by: Ammara Qaisar (DHC-6234)


# Task 1: Term Deposit Subscription Prediction

## Problem Statement

The objective of this project is to predict whether a bank customer will subscribe to a term deposit after a direct marketing campaign.

Machine Learning classification algorithms are used to analyze customer information and predict the subscription outcome.

This project also compares multiple machine learning models and explains predictions using Explainable AI (SHAP).

# Objective

The main objectives of this project are:

- Load and understand the Bank Marketing dataset.
- Perform Exploratory Data Analysis (EDA).
- Clean and preprocess the data.
- Encode categorical variables.
- Train classification models.
- Compare model performance.
- Evaluate using Accuracy, Confusion Matrix, F1-Score and ROC Curve.
- Explain predictions using SHAP.

In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

# Explainable AI
import shap

# Load Dataset

The Bank Marketing dataset is loaded into a pandas DataFrame.

The dataset contains customer demographic information, previous campaign details, and whether the customer subscribed to a term deposit.

In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/bank-additional-full dataset.csv",
    sep=';'
)

In [ ]:
df.head()

# Dataset Overview

Before building machine learning models, it is important to understand the structure of the dataset.

The following analysis includes:

- Number of rows and columns
- Data types
- Missing values
- Duplicate values
- Statistical summary

In [ ]:
print("Shape of Dataset:", df.shape)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
df[df.duplicated()]

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("New Shape:", df.shape)

# Understanding Target Variable

The target variable is **y**, where:

- yes → Customer subscribed to term deposit.
- no → Customer did not subscribe.

Let's examine the distribution of the target variable.

In [ ]:
import csv
from io import StringIO
import pandas as pd

# Read the raw content of the file
with open("/content/drive/MyDrive/bank-additional-full dataset.csv", 'r') as f:
    lines = f.readlines()

# The first line contains the headers
header_line = lines[0].strip()

# Split the header line by semicolon and remove surrounding quotes from each part
column_names = [col.strip().strip('"') for col in header_line.split(';')]

# Prepare a list to hold the parsed data rows
parsed_data = []

# Process data lines by splitting and explicitly stripping quotes from each field
for line in lines[1:]: # Iterate over actual data lines
    # Strip newline characters and then the outermost double quotes from the entire line
    processed_line = line.strip().strip('"')

    # Split the processed line by semicolon
    raw_fields = processed_line.split(';')

    cleaned_fields = []
    for field in raw_fields:
        # Explicitly strip any remaining leading/trailing double quotes from each field
        cleaned_fields.append(field.strip('"'))
    parsed_data.append(cleaned_fields)

# Create DataFrame from the parsed data and column names
df = pd.DataFrame(parsed_data, columns=column_names)

# Explicitly convert numerical columns to numeric types
numeric_cols = ['age', 'duration', 'campaign', 'pdays', 'previous',
                'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

for col in numeric_cols:
    # Use pd.to_numeric with errors='coerce' to handle any non-numeric values gracefully
    # Then, explicitly cast to float to ensure the dtype is numeric even if all values are NaN
    df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)

# Check the 'y' column, which was the original point of failure
df['y'].value_counts()

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(data=df, x='y')

plt.title("Distribution of Target Variable")

plt.xlabel("Subscription")

plt.ylabel("Count")

plt.show()

# Initial Observations

From the above analysis, we can observe:

- The dataset contains more than 41,000 customer records.
- The target variable is imbalanced.
- Most customers did not subscribe to the term deposit.
- There are both numerical and categorical features.
- Further preprocessing is required before model training.

# Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) is performed to understand the characteristics of the dataset.

The main objectives are:

- Understand the distribution of numerical variables.
- Analyze categorical variables.
- Identify relationships between features.
- Detect outliers.
- Discover insights that may improve model performance.

In [ ]:
# Separate numerical and categorical columns

numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns

categorical_columns = df.select_dtypes(include=['object']).columns

print("Numerical Columns:\n", numerical_columns)

print("\nCategorical Columns:\n", categorical_columns)

# Distribution of Customer Age

The age distribution helps us understand the demographic composition of the customers.

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(df['age'], bins=30, kde=True)

plt.title("Age Distribution")

plt.xlabel("Age")

plt.ylabel("Frequency")

plt.show()

# Job Distribution

This visualization shows the number of customers in each occupation category.

In [ ]:
plt.figure(figsize=(12,6))

sns.countplot(data=df, y='job', order=df['job'].value_counts().index)

plt.title("Customer Job Distribution")

plt.show()

# Marital Status Distribution

This chart illustrates the marital status of customers.

In [ ]:
plt.figure(figsize=(7,4))

sns.countplot(data=df, x='marital')

plt.title("Marital Status")

plt.show()

# Education Level

This chart displays the educational qualifications of customers.

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(data=df, y='education')

plt.title("Education Level")

plt.show()

# Housing Loan Status

This chart shows how many customers have housing loans.

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(data=df, x='housing')

plt.title("Housing Loan")

plt.show()

# Personal Loan Status

This graph represents customers who have personal loans.

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(data=df, x='loan')

plt.title("Personal Loan")

plt.show()

# Subscription by Job

This graph compares customer occupations with term deposit subscriptions.

In [ ]:
plt.figure(figsize=(14,6))

sns.countplot(
    data=df,
    y='job',
    hue='y',
    order=df['job'].value_counts().index
)

plt.title("Subscription by Job")

plt.show()

# Subscription by Education

This chart analyzes subscription behavior across education levels.

In [ ]:
plt.figure(figsize=(12,6))

sns.countplot(
    data=df,
    y='education',
    hue='y'
)

plt.title("Subscription by Education")

plt.show()

# Correlation Analysis

Correlation analysis helps identify relationships between numerical variables.

In [ ]:
plt.figure(figsize=(12,10))

corr = df.select_dtypes(include=['int64','float64']).corr()

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.show()

# Boxplots for Outlier Detection

Outliers may affect model performance. Boxplots help visualize extreme values in numerical features.

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(x=df['age'])

plt.title("Age Boxplot")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(x=df['campaign'])

plt.title("Campaign Boxplot")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(x=df['duration'])

plt.title("Call Duration Boxplot")

plt.show()

# Pairwise Relationship Analysis

Pair plots visualize relationships among selected numerical features and the target variable.

In [ ]:
sample_df = df.sample(1000, random_state=42)

sns.pairplot(
    sample_df,
    vars=['age','duration','campaign','previous'],
    hue='y'
)

plt.show()

# Observations from EDA

Key findings from the exploratory analysis include:

- The dataset contains both numerical and categorical variables.
- The target variable is imbalanced, with more customers not subscribing.
- Customers from different occupations and education levels show varying subscription rates.
- Several numerical features contain outliers.
- Correlation among numerical features is generally low to moderate, indicating that multiple variables may contribute independently to the prediction task.

# Data Preprocessing

Data preprocessing is an essential step in machine learning.

The objectives of this stage are:

- Handle missing values.
- Identify unknown values.
- Convert categorical variables into numerical format.
- Separate features and target variable.
- Prepare the dataset for machine learning algorithms.

In [ ]:
# Display first five rows

df.head()

In [ ]:
# Check missing values

df.isnull().sum()

# Checking Missing Values

The Bank Marketing dataset does not contain traditional missing values (NaN).

However, some columns may contain the value **"unknown"**, which represents unavailable information.

These values should be identified before training the model.

In [ ]:
# Count unknown values in each categorical column

for column in df.select_dtypes(include='object').columns:
    print(f"\n{column}")
    print(df[column].value_counts().head())

In [ ]:
# Count all "unknown" values

(df == "unknown").sum()

# Handling Unknown Values

Instead of removing rows containing "unknown", we will keep them as a separate category.

This preserves valuable information and avoids losing a large number of observations.

In [ ]:
# Verify data types

df.dtypes

# Encoding Categorical Features

Machine learning algorithms require numerical input.

Therefore, all categorical variables are converted into numeric labels using Label Encoding.

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

In [ ]:
categorical_columns = df.select_dtypes(include='object').columns

categorical_columns

In [ ]:
for column in categorical_columns:
    df[column] = label_encoder.fit_transform(df[column])

In [ ]:
df.head()

In [ ]:
df.info()

# Separating Features and Target Variable

The dataset contains one target variable:

**y**

where

- 0 = No Subscription
- 1 = Subscription

The remaining columns are used as input features.

In [ ]:
X = df.drop("y", axis=1)

y = df["y"]

In [ ]:
print(X.shape)

print(y.shape)

# Splitting Dataset

The dataset is divided into:

- Training Data (80%)
- Testing Data (20%)

The training data is used for learning, while the testing data is used to evaluate model performance.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training Features :", X_train.shape)

print("Testing Features :", X_test.shape)

print("Training Labels :", y_train.shape)

print("Testing Labels :", y_test.shape)

# Data Preprocessing Summary

The following preprocessing steps have been completed:

- Verified missing values.
- Examined "unknown" categories.
- Encoded categorical variables.
- Separated features and target variable.
- Split the data into training and testing sets.

The dataset is now ready for machine learning model development.

# Model Building

After preprocessing the dataset, machine learning models are developed to predict whether a customer will subscribe to a term deposit.

In this project, the following classification models are implemented:

- Logistic Regression
- Random Forest Classifier

The performance of each model will be evaluated using various evaluation metrics.

# Logistic Regression

Logistic Regression is a supervised machine learning algorithm used for binary classification problems.

It predicts the probability that a customer will subscribe to a term deposit.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
lr.fit(X_train, y_train)

In [ ]:
lr.fit(X_train, y_train)

In [ ]:
lr_predictions = lr.predict(X_test)

# Logistic Regression Evaluation

The Logistic Regression model is evaluated using:

- Accuracy
- Classification Report
- Confusion Matrix
- ROC Curve
- AUC Score

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, lr_predictions)

In [ ]:
print(classification_report(y_test, lr_predictions))

In [ ]:
cm = confusion_matrix(y_test, lr_predictions)

ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')

plt.title("Logistic Regression Confusion Matrix")

plt.show()

In [ ]:
lr_probability = lr.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, lr_probability)

print("AUC Score :", lr_auc)

In [ ]:
fpr,tpr,_ = roc_curve(y_test, lr_probability)

plt.figure(figsize=(8,6))

plt.plot(fpr,tpr,label="Logistic Regression")

plt.plot([0,1],[0,1],'r--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve - Logistic Regression")

plt.legend()

plt.show()

# Random Forest Classifier

Random Forest is an ensemble learning algorithm that combines multiple decision trees to improve prediction accuracy and reduce overfitting.

It is one of the most powerful algorithms for classification problems.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

In [ ]:
rf.fit(X_train,y_train)

In [ ]:
rf.fit(X_train,y_train)

In [ ]:
rf_predictions = rf.predict(X_test)

# Random Forest Evaluation

The Random Forest model is evaluated using the same metrics as Logistic Regression for comparison.

In [ ]:
accuracy_score(y_test,rf_predictions)

In [ ]:
print(classification_report(y_test,rf_predictions))

In [ ]:
cm = confusion_matrix(y_test,rf_predictions)

ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap="Greens")

plt.title("Random Forest Confusion Matrix")

plt.show()

In [ ]:
rf_probability = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test,rf_probability)

print("AUC Score :",rf_auc)

In [ ]:
fpr,tpr,_=roc_curve(y_test,rf_probability)

plt.figure(figsize=(8,6))

plt.plot(fpr,tpr,label="Random Forest")

plt.plot([0,1],[0,1],'r--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve - Random Forest")

plt.legend()

plt.show()

# Model Comparison

The performance of Logistic Regression and Random Forest is compared using Accuracy and AUC Score.

In [ ]:
comparison = pd.DataFrame({

'Model':['Logistic Regression','Random Forest'],

'Accuracy':[

accuracy_score(y_test,lr_predictions),

accuracy_score(y_test,rf_predictions)

],

'AUC Score':[

lr_auc,

rf_auc

]

})

comparison

In [ ]:
comparison.sort_values(by="Accuracy",ascending=False)

In [ ]:
plt.figure(figsize=(6,4))

sns.barplot(data=comparison,x="Model",y="Accuracy")

plt.title("Model Accuracy Comparison")

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

sns.barplot(data=comparison,x="Model",y="AUC Score")

plt.title("Model AUC Score Comparison")

plt.show()

# Model Performance Summary

The results indicate the comparative performance of Logistic Regression and Random Forest.

The model with the higher Accuracy and AUC Score is considered the better-performing model for this dataset.

In most cases, Random Forest achieves better predictive performance because it captures complex non-linear relationships within the data.

# Explainable Artificial Intelligence (SHAP)

Machine learning models often act as black boxes, making it difficult to understand how they make predictions.

To improve transparency and interpretability, SHAP (SHapley Additive exPlanations) is used.

SHAP explains:

- Which features influence predictions.
- Whether a feature increases or decreases the prediction.
- The importance of each feature.
- Individual prediction explanations.

In [ ]:
# Install SHAP (Run only once in Google Colab)

!pip install shap

In [ ]:
import shap

# Create SHAP Explainer

The Random Forest model achieved better performance than Logistic Regression.

Therefore, SHAP explanations are generated using the Random Forest model.

In [ ]:
explainer = shap.TreeExplainer(rf)

In [ ]:
shap_values = explainer.shap_values(X_test)

# Global Feature Importance

The SHAP Summary Plot displays the overall importance of features in predicting customer subscription.

Features with larger SHAP values have greater influence on the model predictions.

In [ ]:
shap.summary_plot(
    shap_values[1],
    X_test
)

# SHAP Bar Plot

The SHAP Bar Plot ranks features according to their average impact on the prediction.

In [ ]:
shap.summary_plot(
    shap_values[1],
    X_test,
    plot_type="bar"
)

# Explaining Individual Predictions

The following waterfall plots explain why the model predicted the output for five individual customers.

Each feature either increases or decreases the prediction probability.

In [ ]:
for i in range(5):

    shap.plots.waterfall(

        shap.Explanation(

            values=shap_values[1][i],

            base_values=explainer.expected_value[1],

            data=X_test.iloc[i],

            feature_names=X.columns

        )

    )

# Feature Importance

The Random Forest model also provides feature importance scores.

These scores indicate which variables contribute the most to prediction.

In [ ]:
feature_importance = pd.DataFrame({

'Feature':X.columns,

'Importance':rf.feature_importances_

})

In [ ]:
feature_importance = feature_importance.sort_values(

by="Importance",

ascending=False

)

feature_importance.head(10)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(

data=feature_importance.head(10),

x="Importance",

y="Feature"

)

plt.title("Top 10 Important Features")

plt.show()

# Final Results

The performance of the machine learning models was compared using multiple evaluation metrics.

Among the tested models, Random Forest achieved the highest predictive performance and produced more reliable classification results.

SHAP analysis provided transparency by identifying the most influential features affecting customer subscription decisions.

# Conclusion

In this project, a machine learning solution was developed to predict whether a bank customer would subscribe to a term deposit.

The project included:

- Data loading and preprocessing
- Exploratory Data Analysis (EDA)
- Feature encoding
- Logistic Regression
- Random Forest Classification
- Model evaluation using Accuracy, Confusion Matrix, Precision, Recall, F1-Score, ROC Curve, and AUC Score
- Explainable AI using SHAP

The Random Forest model outperformed Logistic Regression and demonstrated strong predictive capability.

SHAP explanations enhanced the interpretability of the model by illustrating how different customer attributes influenced individual predictions.

This project demonstrates the practical application of supervised machine learning and explainable artificial intelligence in solving real-world marketing prediction problems.

# Future Work

Future improvements to this project may include:

- Hyperparameter tuning using GridSearchCV or RandomizedSearchCV.
- Handling class imbalance using SMOTE.
- Training advanced models such as XGBoost, CatBoost, and LightGBM.
- Deploying the trained model using Streamlit.
- Monitoring model performance with real-time data.